# Writing a quantum program - addressing bits

Qualtran lets you write quantum programs and subroutines by composing lower-level subroutines, gates, and operations. We call these composable objects *bloqs* because they are the quantum building blocks of a complex algorithm. Composition is recursive: composing (lower-level) bloqs defines (higher-level) bloqs. 

In this tutorial, you will write a very simple quantum program by composing bloqs. The program will declare two input/output registers named 'x' and 'y' and swap the (quantum) integers stored within. Specifically, the quantum integer provided as input to 'x' will be moved to 'y', and vice versa.

## Using `BloqBuilder`

Before we write our simple `swap` program, we'll write the simplest possible program:
let's create a program that takes two integers and does nothing to them. In this code snippet, we

 - import `BloqBuilder`. We'll use methods on this class to construct our program.
 - import some data types. Here, we import `QUInt` which specifies a quantum unsigned integer, as well as `QBit` which specifies a single quantum bit. These are quantum data type *classes*. When writing a quantum program, we need to instantiate them into quantum data type *objects* by providing any *compile-time classical parameters* to the data type's constructor.
 - declare *registers* for our program. We use the `add_register` method. We provide a name for the register; and we provide the quantum data type. The method returns a handle to the declared register that we can use later. These handles are called *soquets*. They will always be instantiated and consumed by the framework&mdash;the programmer should never concern themselves with the data members of a Python soquet object.
 - immediately finish the program. Since this first program does nothing, we immediately return. Our call to `bb.finalize` maps output register names to soquets.  

In [ ]:
# HACK HACK HACK
from qualtran import BloqBuilder
BloqBuilder.add_register = BloqBuilder.add_register_from_dtype

# HACK HACK HACK
import qualtran.testing as qlt_testing
qlt_testing.assert_valid_program = qlt_testing.assert_valid_cbloq

# TODO:
# consdier make soquet data members private.

In [ ]:
# Quantum program 1
# This program does nothing
# 
# Registers:
#   x: an 8-bit quantum unsigned integer
#   y: an 8-bit quantum unsigned integer

from qualtran import BloqBuilder
from qualtran import QUInt, QBit

# Start program construction
bb = BloqBuilder()

# Add input/output registers named 'x' and 'y'
x = bb.add_register('x', QUInt(8))
y = bb.add_register('y', QUInt(8))

program = bb.finalize(x=x, y=y)

At a very basic level, we want the structure of our program to be valid. We can use the following check to do some basic assertions about the structure of the program.

In [ ]:
qlt_testing.assert_valid_program(program)

We can also show a directed acyclic graph representation of the program, which is similarly simple.

In [ ]:
from qualtran.drawing import show_bloq
show_bloq(program)

## Using bloqs

If we want our program to do something, we need to add calls to quantum subroutines. We'll compose `BitSwap` bloqs from the Qualtran standard library to perform the swap of two integers. Let's take a look at the reference documentation for `BitSwap`.

In [ ]:
# TODO: rename; maybe QBitSwap
from qualtran.bloqs.basic_gates import TwoBitSwap as BitSwap

In [ ]:
import sys
sys.path.append('../../dev_tools/')


In [ ]:
from qualtran_dev_tools.parse_docstrings import get_markdown_docstring_lines, get_markdown_docstring
from IPython.display import Markdown

In [ ]:
display(Markdown('\n'.join(['### Reference documentation for `BitSwap`:'] + get_markdown_docstring(BitSwap))))

Unsurprisingly, the bloq promises to swap two bits. Importantly, we see that it has two registers named 'x' and 'y', which we will need when using `BloqBuilder.add(...)` to call this bloq in our program.

## Writing the program

Pseudocode:

```pseudocode
    input quint x, y
    for each bit in x, y:
        x[i], y[i] = bit_swap(x[i], y[i])
    output quint x, y
```

We can translate this pseudocode into a well-formed Qualtran program relatively straightforwardly. We need to take particular care when iterating over "each bit in x, y". What does this actually mean? In Qualtran, we assume that data is encoded in qubits, and writing low-level operations requires temporarily removing the quantum data type abstraction to operate on the bits directly. The programmer must explicitly state their intent to do so by including *bookkeeping* operations in the program. Below, we use `split` and `join` to manipulate individual bits.

`split` takes one `QUInt` soquet (i.e. one handle to a quantum integer) and returns $n$ `QBit` soquets which can be individually manipulated.

In [ ]:
# Quantum program 2: QUIntSwap
# This program swaps two 8-bit integers, x <--> y.
# 
# Registers:
#   x: an 8-bit quantum unsigned integer
#   y: an 8-bit quantum unsigned integer

# Start writing our program.
from qualtran import BloqBuilder
from qualtran import QUInt, QBit
bb = BloqBuilder()

# Add input/output registers named 'x' and 'y'.
n = 4
x = bb.add_register('x', QUInt(n))
y = bb.add_register('y', QUInt(n))

# Split integers into their individual bits
xs = bb.split(x)
ys = bb.split(y)

# Perform BitSwap on each bit
for i in range(n):
    xs[i], ys[i] = bb.add(BitSwap(), x=xs[i], y=ys[i])

# Join bits back to a quantum integer
x = bb.join(xs, QUInt(n))
y = bb.join(ys, QUInt(n))

# Finish up: map final soquets to output register names.
program = bb.finalize(x=x, y=y)

In [ ]:
qlt_testing.assert_valid_program(program)

In [ ]:
show_bloq(program)

## Testing classical logic

Testing quantum programs is challenging in general because of the potentially exponential number of 'paths' connecting inputs and outputs. `BitSwap` and our program only operates classically: if given a classical input (also known as a *computaitonal basis state*) it will return one and only one classical output. We can exploit this when testing bloqs that encode classical-reversible logic. Here, we call our program with `x=5, y=6` and note that the returned values are indeed swapped.

In [ ]:
program.call_classically(x=5, y=6)